## 1. Perbandingan Geographically Weighted Regression, Decision Tree, dan K-Nearest Neighbors dalam Prediksi Pendapatan per Kapita di Chicago

| Komponen | Variabel yang dipakai |
|---|---|
| **Y** | Pendapatan per kapita (`PerCInc14`). |
| **X₁** | Tingkat pengangguran (`Unemp14 / LaborFrc`), yang mewakili kondisi pasar kerja. |
| **X₂** | Proporsi orang dewasa tanpa ijazah SMA (`NoHS14` dibagi total lima kategori pendidikan dari `NoHS14` sampai `ClgGrad14`, termasuk kategori gelar pascasarjana). Ini mewakili tingkat pendidikan wilayah. |
| **Lokasi** | Titik pusat tiap *community area*, dihitung dari geometri wilayah untuk pembobotan GWR. |

dataset : https://geodacenter.github.io/data-and-lab/comarea_vars

## 2. Load Dataset

Membaca shapefile community area Chicago dengan geopandas, memastikan jumlah baris dan kolom yang dibutuhkan sesuai, lalu mereproyeksikan geometri ke EPSG:32616 (meter) supaya centroid bisa dihitung dalam satuan jarak nyata.

In [1]:
import geopandas as gpd

gdf = gpd.read_file("dataset/ComArea_ACS14_f.shp")
assert len(gdf) == 77, len(gdf)
assert {"ComAreaID", "community", "PerCInc14", "Unemp14", "LaborFrc",
        "NoHS14", "HSGrad14", "SmClg14", "ClgGrad14"} <= set(gdf.columns), sorted(gdf.columns)
gdf = gdf.to_crs(32616)
gdf[["ComAreaID", "community", "PerCInc14", "Unemp14", "LaborFrc"]].head()

,ComAreaID,community,PerCInc14,Unemp14,LaborFrc
0,35,DOUGLAS,215254,1620,8869
1,36,OAKLAND,54204,865,3139
2,37,FULLER PARK,32828,378,1036
3,38,GRAND BOULEVARD,293535,2619,10756
4,39,KENWOOD,261582,1304,9283


## 3. Feature Engineering

Catatan: spesifikasi menyebut lima kategori pendidikan; dataset ACS ini hanya memuat empat (NoHS14, HSGrad14, SmClg14, ClgGrad14), jadi X2 memakai jumlah keempatnya.

Membentuk variabel model: X1 = rasio pengangguran, X2 = proporsi tanpa ijazah SMA, Y = pendapatan per kapita, serta centroid x_m/y_m; assertion memastikan tidak ada nilai kosong dan kedua prediktor berada di rentang 0 sampai 1.

In [2]:
df = gdf.copy()
df["x1_unemp_rate"] = df["Unemp14"] / df["LaborFrc"]
df["x2_no_hs"] = df["NoHS14"] / df[["NoHS14", "HSGrad14", "SmClg14", "ClgGrad14"]].sum(axis=1)
df["y"] = df["PerCInc14"]
centroids = df.geometry.centroid
df["x_m"] = centroids.x
df["y_m"] = centroids.y

assert df[["x1_unemp_rate", "x2_no_hs", "y", "x_m", "y_m"]].isna().sum().sum() == 0
assert df["x1_unemp_rate"].between(0, 1).all()
assert df["x2_no_hs"].between(0, 1).all()
df[["community", "y", "x1_unemp_rate", "x2_no_hs", "x_m", "y_m"]].describe()

,y,x1_unemp_rate,x2_no_hs,x_m,y_m
count,7.700000e+01,77.000000,77.000000,77.000000,7.700000e+01
mean,2.895573e+05,0.158927,0.192422,443958.997676,4.632167e+06
std,3.600327e+05,0.084881,0.114642,5787.984034,1.106758e+04
min,1.490500e+04,0.042418,0.017937,425957.828150,4.612218e+06
25%,1.130910e+05,0.093559,0.108831,440201.764983,4.623674e+06
50%,1.903820e+05,0.140004,0.168188,444506.602087,4.631112e+06
75%,2.861380e+05,0.222155,0.265260,448510.533181,4.641983e+06
max,2.075859e+06,0.404459,0.545832,455498.117941,4.651062e+06


## 4. Train/Test Split

Membagi 77 baris menjadi 61 data latih dan 16 data uji dengan random_state=42, menyiapkan matriks fitur, target, dan koordinat untuk GWR, lalu mendefinisikan evaluate() untuk menghitung R2, RMSE, dan MAE.

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FEATURES = ["x1_unemp_rate", "x2_no_hs"]
idx_train, idx_test = train_test_split(df.index, test_size=0.2, random_state=42)
X_train, X_test = df.loc[idx_train, FEATURES].to_numpy(), df.loc[idx_test, FEATURES].to_numpy()
y_train, y_test = df.loc[idx_train, "y"].to_numpy(), df.loc[idx_test, "y"].to_numpy()
coords_train = df.loc[idx_train, ["x_m", "y_m"]].to_numpy()
coords_test = df.loc[idx_test, ["x_m", "y_m"]].to_numpy()

def evaluate(name, y_true, y_pred):
    y_pred = np.asarray(y_pred).ravel()
    return {"Model": name,
            "R2": r2_score(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAE": mean_absolute_error(y_true, y_pred)}

results = {}
assert len(idx_train) == 61 and len(idx_test) == 16
assert X_train.shape == (61, 2) and X_test.shape == (16, 2)
print(idx_test.sort_values().tolist())

[0, 4, 10, 12, 18, 28, 33, 35, 45, 47, 50, 53, 58, 66, 68, 74]


## 5. Model 1: Geographically Weighted Regression

Memilih bandwidth adaptive (kernel bisquare), melatih GWR pada data latih, memprediksi 16 lokasi uji di luar sampel, dan menyimpan metriknya; bila pencarian bandwidth gagal, model jatuh ke bandwidth fixed.

In [4]:
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

y_train_col = y_train.reshape(-1, 1)
selector = Sel_BW(coords_train, y_train_col, X_train, fixed=False, kernel="bisquare")
try:
    gwr_bw = selector.search()
    gwr_fixed = False
except Exception as exc:
    print("adaptive bandwidth failed:", exc)
    gwr_bw, gwr_fixed = 30, True
    selector = Sel_BW(coords_train, y_train_col, X_train, fixed=gwr_fixed, kernel="bisquare")
    gwr_bw = selector.search(bw_min=2)

gwr_model = GWR(coords_train, y_train_col, X_train, gwr_bw,
                fixed=gwr_fixed, kernel="bisquare")
gwr_res = gwr_model.fit()
gwr_pred = gwr_model.predict(coords_test, X_test).predictions.ravel()

results["GWR"] = evaluate("GWR", y_test, gwr_pred)
print(f"bandwidth={gwr_bw} fixed={gwr_fixed} in-sample R2={gwr_res.R2:.4f}")
results["GWR"]

bandwidth=47.0 fixed=False in-sample R2=0.5364


{'Model': 'GWR',
 'R2': -1.1886962485990469,
 'RMSE': 303406.6992284818,
 'MAE': 229210.67670426634}

## 6. Model 2: Decision Tree

Melatih Decision Tree dengan hyperparameter default pada data latih, lalu menyimpan R2, RMSE, dan MAE hasil prediksi pada data uji.

In [5]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
results["Decision Tree"] = evaluate("Decision Tree", y_test, tree.predict(X_test))
results["Decision Tree"]

{'Model': 'Decision Tree',
 'R2': -0.23089091761321368,
 'RMSE': 227531.85243004988,
 'MAE': 168308.75}

## 7. Model 3: K-Nearest Neighbors

Menstandarkan fitur memakai statistik data latih saja (mencegah kebocoran), melatih KNN dengan k=5, lalu menyimpan metriknya pada data uji.

In [6]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_scaled, X_test_scaled = scaler.transform(X_train), scaler.transform(X_test)

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
results["KNN"] = evaluate("KNN", y_test, knn.predict(X_test_scaled))
results["KNN"]

{'Model': 'KNN',
 'R2': -0.752853601533567,
 'RMSE': 271522.13612017897,
 'MAE': 186761.52500000002}

## 8. Hasil Perbandingan

Menyusun tabel perbandingan ketiga model terurut menurut RMSE, memastikan semua metrik terisi, mencetak rata-rata Y dan nama model terbaik, lalu menampilkan tabelnya.

In [7]:
comparison = pd.DataFrame(results.values()).sort_values("RMSE").reset_index(drop=True)
assert set(comparison["Model"]) == {"GWR", "Decision Tree", "KNN"}
assert comparison[["R2", "RMSE", "MAE"]].notna().all().all()
baseline_rmse = mean_squared_error(y_test, np.full(len(y_test), y_train.mean())) ** 0.5
print(f"Baseline (train mean) RMSE = {baseline_rmse:.1f}")
print(f"PerCInc14 mean = {df['y'].mean():.1f} (data units, not re-scaled)")
print(f"Best model by RMSE: {comparison.loc[0, 'Model']}")
comparison

Baseline (train mean) RMSE = 213818.2
PerCInc14 mean = 289557.3 (data units, not re-scaled)
Best model by RMSE: Decision Tree


,Model,R2,RMSE,MAE
0,Decision Tree,-0.230891,227531.852430,168308.750000
1,KNN,-0.752854,271522.136120,186761.525000
2,GWR,-1.188696,303406.699228,229210.676704


**Kesimpulan:** Decision Tree memberi RMSE terkecil (227.532), unggul 43.990 dari KNN (271.522) dan 75.875 dari GWR (303.407).

R² ketiga model bernilai negatif (Decision Tree -0.231, KNN -0.753, GWR -1.189), artinya pada 16 baris uji prediksi masih lebih buruk daripada sekadar memakai nilai rata-rata. Penyebab utamanya sebaran `PerCInc14` sangat ekstrem: data uji memuat community area dengan nilai 698.569 dan 795.576, sementara model hanya memakai dua prediktor (tingkat pengangguran dan proporsi tanpa ijazah SMA). GWR juga kalah di luar sampel meski R² in-sample-nya 0.536, karena pembobotan spasialnya diestimasi dari 61 titik latih.

Skala Y mengikuti satuan asli dataset (rata-rata 289.557, kemungkinan persepuluh dolar); tidak ada rescale yang diterapkan.

Sebagai pembanding, memprediksi rata-rata data latih untuk semua baris uji menghasilkan RMSE 213.818, lebih baik daripada ketiga model di atas; artinya belum ada model yang layak dipakai untuk prediksi di luar sampel.